In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import os
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from phd_project.config import config
cfg = config.load_config()

In [9]:
ROOT = cfg["analysis_data"]["dc2_sdof_fitting"]

In [10]:
# mapping parameters from design output to columns for df
design_out_parameter_map = {
    "roof_height": "['structure']['level_coordinates'][-1]",
    "V_wind_GQWSI_LC1": "['nonseismic_design_outputs']['GQWSI_LC1']['uls_wind_base_shear']",
    "V_wind_GQWSI_LC2": "['nonseismic_design_outputs']['GQWSI_LC2']['uls_wind_base_shear']",
    "V_wind_GQWSI_LC3": "['nonseismic_design_outputs']['GQWSI_LC3']['uls_wind_base_shear']",
    "T1_GQWSI_LC1": "['nonseismic_design_outputs']['GQWSI_LC1']['period']",
    "T1_GQWSI_LC2": "['nonseismic_design_outputs']['GQWSI_LC2']['period']",
    "T1_GQWSI_LC3": "['nonseismic_design_outputs']['GQWSI_LC3']['period']",
    "gravity_frame_elastic_baseshear": "['seismic_design_outputs']['gravity_frame_elastic_baseshear']",
    "gravity_design_equivalent_seismic_baseshear": "['seismic_design_outputs']['gravity_design_equivalent_seismic_baseshear']",
    "seismic_mass":"['seismic_design_outputs']['seismic_mass']",
    "ductility_class":"['seismic_design_outputs']['ductility_class']",    
    "vertical_regularity":"['seismic_design_outputs']['vertical_regularity']",
    "q_design":"['seismic_design_outputs']['q_design']",
    "q_D":"['seismic_design_outputs']['q_D']",
    "q_R":"['seismic_design_outputs']['q_R']",
    "q_S":"['seismic_design_outputs']['q_S']",
    "q_max":"['seismic_design_outputs']['q_max']",
    "design_period":"['seismic_design_outputs']['design_period']",
    "design_spectral_acceleration":"['seismic_design_outputs']['design_spectral_acceleration']",
    "design_baseshear":"['seismic_design_outputs']['design_baseshear']",
    "lambda":"['seismic_design_outputs']['lambda']",
    "S_alpha_RP":"['seismic_design_outputs']['spectrum_parameters']['S_alpha_RP']",
    "S_beta_RP":"['seismic_design_outputs']['spectrum_parameters']['S_beta_RP']",
    "S_alpha":"['seismic_design_outputs']['spectrum_parameters']['S_alpha']",
    "S_beta":"['seismic_design_outputs']['spectrum_parameters']['S_beta']",
    "T_beta":"['seismic_design_outputs']['spectrum_parameters']['T_beta']",
    "T_A":"['seismic_design_outputs']['spectrum_parameters']['T_A']",
    "T_B":"['seismic_design_outputs']['spectrum_parameters']['T_B']",
    "T_C":"['seismic_design_outputs']['spectrum_parameters']['T_C']",
    "T_D":"['seismic_design_outputs']['spectrum_parameters']['T_D']",
    "F_alpha":"['seismic_design_outputs']['spectrum_parameters']['F_alpha']",
    "F_beta":"['seismic_design_outputs']['spectrum_parameters']['F_beta']",
    "F_T":"['seismic_design_outputs']['spectrum_parameters']['F_T']",
    "F_A":"['seismic_design_outputs']['spectrum_parameters']['F_A']",
    "site_category":"['seismic_design_outputs']['spectrum_parameters']['site_category']",
    "S_delta":"['seismic_design_outputs']['spectrum_parameters']['S_delta']",
    "delta":"['seismic_design_outputs']['spectrum_parameters']['delta']",
    "seismic_action_class":"['seismic_design_outputs']['spectrum_parameters']['seismic_action_class']"
}

In [11]:
def read_out_value(data, path):
    try:
        return eval("data" + path)
    except:
        return np.nan

In [12]:
buildings = os.listdir(ROOT)
design_data = {}
for building in buildings:
    with open(ROOT / building / f"{building}_designfile.json", "r") as f:
        design_data[building] = json.load(f)

building_data_dicts = []
for name, data in design_data.items():
    building_data = {}
    building_data["name"] = name
    building_data["n_storeys"] = int(name[0])
    for df_tag, dd_path in design_out_parameter_map.items():
        building_data[df_tag] = read_out_value(data, dd_path)

    # get the design storey forces
    storey_forces = data["seismic_design_outputs"]["storey_forces"]
    storey_shears = np.cumsum(storey_forces)
    for ii in range(7):
        if ii+1 <= len(storey_forces):
            building_data[f"seismic_F{ii+1}"] = storey_forces[ii]
        else:
            building_data[f"seismic_F{ii+1}"] = np.nan

    for ii in range(7):
        if ii+1 <= len(storey_shears):
            building_data[f"seismic_V{ii+1}"] = storey_shears[ii]
        else:
            building_data[f"seismic_F{ii+1}"] = np.nan

    building_data_dicts.append(building_data)

building_data_df = pd.DataFrame(building_data_dicts)
building_data_df = building_data_df.sort_values(by=["n_storeys", "S_alpha_RP"])
building_data_df = building_data_df.reset_index()

with open(cfg["proc_data"]["dc2_casestudy_dataset"], "wb") as f:
    pickle.dump(building_data_df, f)